# finance / household / runrate

The normal annual spend with the one-offs taken out, and the one-off allowance.

| Term | In This Notebook |
|---|---|
| **Intent** | Separate the household's steady spending from its irregular spending, to give one baseline annual figure that the cost growth rate is applied to. Placeholder: a synthetic monthly series with three one-offs |
| **Project Flavour** | `static`, local to the notebook (`model.md` 3.2) |
| **Data Splits** | `training` twenty four months of synthetic spending with three one-offs, until the `bank` table exists; `golden` the exact request that becomes `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, target, dataset names and signature names as constants (`model.md` 6.6).

In [ ]:
import json

from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "finance/household/runrate"
MODEL_NAME = "finance_household_runrate"
TARGETS = [{"kind": "local"}]
TRAINING = "household_runrate_training_2026-11-05_2026-11-05"
GOLDEN = "household_runrate_golden"
INPUT = "finance_household_runrate_in"
OUTPUTS = ["finance_household_runrate_normal_spend_dollars", "finance_household_runrate_oneoff_allowance_dollars"]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training` file only when it is absent (`model.md` 6.5). The source is synthetic until the real series is loaded, so no query runs.

In [ ]:
DEVIATIONS = [-300.0, 200.0, -100.0, 300.0, -200.0, 0.0, 100.0, -300.0, 200.0, -100.0, 300.0, -200.0, 100.0, 0.0, -200.0, 300.0, -100.0, 200.0, -300.0, 100.0, 0.0, 200.0, -100.0, -200.0]
ONEOFFS = {5: 6000.0, 14: 9000.0, 20: 4000.0}

training_pl, training_path = ensure_data(
    WORKSPACE,
    "training",
    TRAINING,
    lambda: {"month": list(range(24)), "spend_dollars": [10000.0 + deviation + ONEOFFS.get(index, 0.0) for index, deviation in enumerate(DEVIATIONS)]},
)
print("training", training_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** flag a month more than five median absolute deviations above the median as a one-off, on `training` only (R6), and take the run rate from the rest. The threshold is the first thing the real data decides.

In [ ]:
import numpy as np

spend = training_pl["spend_dollars"].to_numpy()
median = float(np.median(spend))
mad = float(np.median(np.abs(spend - median)))
oneoff = spend > median + 5.0 * mad
CONSTANTS = dict(zip(OUTPUTS, [round(float(spend[~oneoff].mean()) * 12.0, 0), round(float((spend[oneoff] - median).sum()) / (len(spend) / 12.0), 0)], strict=True))
print("one-off months", np.flatnonzero(oneoff).tolist(), "constants", CONSTANTS)
assert np.flatnonzero(oneoff).tolist() == [5, 14, 20], "the rule did not find the one-offs"

## Package

- **Stage:** 5, package.
- **Intent:** write the golden rows, build the signature, write a `static` project whose constants are the fitted values, verify it and replay the golden through the library loader (R1, R3).

In [ ]:
from workbench.model import build_signature, package_project, signature_input, signature_output
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {INPUT: [1.0, None], "expect": ["ok", "error"]})

signature = build_signature(
    MODEL_NAME,
    [signature_input(INPUT, "none", "Unused input.")],
    [
        signature_output("finance_household_runrate_normal_spend_dollars", "none", "Normal annual spend with the one-offs taken out, dollars.", datatype="FP64"),
        signature_output("finance_household_runrate_oneoff_allowance_dollars", "none", "Average annual spend on one-offs, dollars.", datatype="FP64"),
    ],
)
project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic monthly spending with three one-offs, training split",
        [training_path, golden_path],
        "median absolute deviation rule for one-offs",
    ),
    constants=CONSTANTS,
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped loader")